In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from matplotlib.patches import Circle
from scipy.stats import linregress
import matplotlib as mpl
from matplotlib.gridspec import GridSpec
from astropy import units as u
from pylab import *
from matplotlib.patches import Ellipse
from radio_beam import Beam
import astropy.units as u
from matplotlib.patches import Rectangle

In [ ]:
P_thr   = 0.1 # K
dRM_thr = 150 # rad/m^2

In [ ]:
def read_files_4channels(directory,stokes,filetype):

    band = ['A','B','C','D']
    data_list = []
    hdr_list = []
    print('Reading in '+stokes+' for filetype: '+filetype)

    for i in range(0,4):
        print('band '+band[i])
        hdu = fits.open(directory+stokes+band[i]+'_'+filetype+'.fits')
        data_list.append(hdu[0].data)

        hdr = fits.Header()
        for card in hdu[0].header.cards:
            if card.keyword.strip() != "":
                hdr.append(card)
        hdr['OBJECT'] = stokes+band[i]+'_'+filetype
        hdr_list.append(hdr)
        #print(repr(hdr))
        #print('-------------------')

    gc.collect()

    return data_list,hdr_list

In [ ]:
def make_landecker_map(data1,data2,hdr,v1max=300,v2max=1,
                       cmap1='RdBu_r',cmap2 = 'viridis',
                       llim = [192,52], blim = [-7,10],filename='test',
                       *args,**kwargs):
    
    aspect = (blim[1]-blim[0])/(llim[0]-llim[1])
    print(aspect)
    
    c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
    fs = 22
    
    fig = plt.figure(figsize=(16,11.3))
    
    plt.subplots_adjust(hspace=0.0,left=0.08, right=0.98, top=0.99, bottom=0.08)
    
    cmap = mpl.colormaps.get_cmap(cmap1)  # viridis is the default colormap for imshow
    cmap.set_bad(color='grey')
    
    ax1  = fig.add_subplot(211, projection=WCS(hdr).celestial)
    im1  = ax1.imshow(data1, origin='lower', vmin=-v1max, vmax=v1max,cmap=cmap)
    ax1.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax1.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    #ax1.set_xticks([125,130,135])
    cbar1 = fig.colorbar(im1, ax=ax1, orientation='vertical',fraction=0.1,pad=0.0,aspect=15)
    cbar1.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    cbar1.set_ticks([-200,-100,0,100,200])

    cmap = mpl.colormaps.get_cmap(cmap2)  # viridis is the default colormap for imshow
    cmap.set_bad(color='grey')
   
    ax2  = fig.add_subplot(212, projection=WCS(hdr).celestial)
    im2  = ax2.imshow(data2, origin='lower', vmin=0, vmax=v2max,cmap=cmap)
    ax2.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax2.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    #ax2.set_xticks([125,130,135])
    cbar2 = fig.colorbar(im2, ax=ax2, orientation='vertical',fraction=0.1,pad=0.0,aspect=15)
    cbar2.set_label(r'PI (K)', fontsize=fs)
    cbar2.set_ticks([0,0.1,0.2,0.3,0.4,0.5])
    
    
    ax2.set_xlabel('Galactic Longitude',fontsize=fs)
    fig.text(0.02,0.45,'Galactic Latitude',fontsize=fs,rotation='vertical')
    for ax in [ax1,ax2]:
        ax.tick_params(axis='both', labelsize=fs)
        ax.set_ylabel('  ',fontsize=fs)
        ax.tick_params(axis='both', which='both', width=2, length=6)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        
    for cbar in [cbar1,cbar2]:
        cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)
    
    #plt.savefig('/home/aordog/CGPS_GMIMS_PLOTS/'+filename+'.pdf')
    plt.savefig('../plots/maps/'+filename+'.pdf')

    return

In [ ]:
# Read in RM, Pearson R, and standard error in RM:
hdu_RM_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_CG_conv4_regrd.fits')
RM_CG_all = hdu_RM_CG[0].data
RM_CG     = RM_CG_all.copy()
rvalue_CG = hdu_RM_CG[2].data
stderr_CG = hdu_RM_CG[4].data
hdr       = hdu_RM_CG[0].header

hdu_RM_G = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_G_regrd.fits')
RM_G_all = hdu_RM_G[0].data
RM_G     = RM_G_all.copy()
rvalue_G = hdu_RM_G[2].data
stderr_G = hdu_RM_G[4].data

hdu_RM_C = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_C_conv4_regrd.fits')
RM_C_all = hdu_RM_C[0].data
RM_C     = RM_C_all.copy()
rvalue_C = hdu_RM_C[2].data
stderr_C = hdu_RM_C[4].data

# Read in polarised intensity:
hdu_PI_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_CG_conv4_regrd_PI_of_mean.fits')
PI_CG     = hdu_PI_CG[0].data

hdu_PI_G = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_G_regrd_PI_of_mean.fits')
PI_G     = hdu_PI_G[0].data 

hdu_PI_C = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_C_conv4_regrd_PI_of_mean.fits')
PI_C     = hdu_PI_C[0].data 

# Set outside of mosaics to NaN:
RM_CG[RM_CG_all==0.0]     = np.nan
rvalue_CG[RM_CG_all==0.0] = np.nan
stderr_CG[RM_CG_all==0.0] = np.nan
PI_CG[RM_CG_all==0.0]     = np.nan

RM_C[RM_C_all==0.0]     = np.nan
rvalue_C[RM_C_all==0.0] = np.nan
stderr_C[RM_C_all==0.0] = np.nan
PI_C[RM_C_all==0.0]     = np.nan


In [ ]:
l = WCS(hdr).all_pix2world(range(RM_CG.shape[1]) ,0, 0)[0]
b = WCS(hdr).all_pix2world(0, range(RM_CG.shape[0]), 0)[1]

RM_CG_filt = RM_CG.copy()
RM_G_filt = RM_G.copy()
RM_C_filt = RM_C.copy()
########################################
RM_CG_filt[PI_CG < P_thr] = np.nan
RM_CG_filt[stderr_CG > dRM_thr] = np.nan

RM_G_filt[PI_G < P_thr] = np.nan
RM_G_filt[stderr_G > dRM_thr] = np.nan

RM_C_filt[PI_C < P_thr] = np.nan
RM_C_filt[stderr_C > dRM_thr] = np.nan
########################################

idx_bad = np.where((l>179.25) & (l<180.75))

RM_CG_filt[:,idx_bad] = np.nan
PI_CG[:,idx_bad] = np.nan
RM_G_filt[:,idx_bad] = np.nan
RM_C_filt[:,idx_bad] = np.nan

In [ ]:
def make_4panel(llim=[192, 52], blim=[-7, 10], filename='test', l_ticks=[50]):
    
    c = SkyCoord(llim, blim, frame='galactic', unit='deg')
    fs = 22

    # Define the overall figure and adjust its size
    #fig = plt.figure(figsize=(20.75, 12))
    fig = plt.figure(figsize=(20.75/3, 12/3))
    
    # Use GridSpec to organize the layout
    # Adjust width_ratios to ensure the colorbars are spaced correctly
    gs = GridSpec(2, 4, figure=fig, width_ratios=[48, 48, 0.5, 2.5], height_ratios=[1, 1], wspace=0.0, hspace=0.0)

    # Create axes for each subplot
    ax1 = fig.add_subplot(gs[0, 0], projection=WCS(hdr).celestial)
    ax2 = fig.add_subplot(gs[0, 1], projection=WCS(hdr).celestial)
    ax3 = fig.add_subplot(gs[1, 0], projection=WCS(hdr).celestial)
    ax4 = fig.add_subplot(gs[1, 1], projection=WCS(hdr).celestial)

    cmap1 = mpl.colormaps.get_cmap('RdBu_r')
    cmap2 = mpl.colormaps.get_cmap('gist_heat_r')
    cmap1.set_bad(color='grey')
    cmap2.set_bad(color='grey')

    im1 = ax1.imshow(RM_G, origin='lower', vmin=-300, vmax=300, cmap=cmap1)
    im2 = ax2.imshow(RM_CG_filt, origin='lower', vmin=-300, vmax=300, cmap=cmap1)
    im3 = ax3.imshow(PI_G, origin='lower', vmin=0, vmax=0.6, cmap=cmap2)
    im4 = ax4.imshow(PI_CG, origin='lower', vmin=0, vmax=0.6, cmap=cmap2)

    # Create colorbars
    cbar2 = fig.colorbar(im2, ax=[ax2], cax=fig.add_subplot(gs[0, 3]), orientation='vertical')
    cbar2.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    cbar2.set_ticks([-200, -100, 0, 100, 200])

    cbar4 = fig.colorbar(im4, ax=[ax4], cax=fig.add_subplot(gs[1, 3]), orientation='vertical')
    cbar4.set_label(r'PI (K)', fontsize=fs)
    cbar4.set_ticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])

    axes = [ax1, ax2, ax3, ax4]
    for i, ax in enumerate(axes):
        ax.set_xlim(WCS(hdr).world_to_pixel(c)[0])
        ax.set_ylim(WCS(hdr).world_to_pixel(c)[1])
        ax.tick_params(axis='both', labelsize=fs)
        ax.set_ylabel('  ', fontsize=fs)
        ax.set_xlabel('  ', fontsize=fs)
        ax.tick_params(axis='both', which='both', width=2, length=6)
        ax.coords[0].set_major_formatter('dd')
        ax.coords[1].set_major_formatter('dd')
        ax.coords[0].set_ticks(l_ticks*u.degree)

        if i in [0, 1]:
            ax.coords[0].set_ticks_visible(False)
            ax.coords[0].set_ticklabel_visible(False)
        if i in [1, 3]:
            ax.coords[1].set_ticks_visible(False)
            ax.coords[1].set_ticklabel_visible(False)

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)

    for cbar in [cbar2, cbar4]:
        cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)

    fig.text(0.5, 0.04, 'Galactic Longitude', fontsize=fs, ha='center')
    fig.text(0.07, 0.5, 'Galactic Latitude', fontsize=fs, va='center', rotation='vertical')

    plt.savefig('../plots/talks/' + filename + '.png', bbox_inches='tight', pad_inches=0)
    #plt.show()

    return


In [ ]:
def make_4panel_v2(llim=[192, 52], blim=[-7, 10], filename='test', l_ticks=[50]):
    
    c = SkyCoord(llim, blim, frame='galactic', unit='deg')
    fs = 10

    # Define the overall figure and adjust its size
    #fig = plt.figure(figsize=(20.75, 12))
    fig = plt.figure(figsize=(20.75/2.5, 12/2.5))
    
    # Use GridSpec to organize the layout
    # Adjust width_ratios to ensure the colorbars are spaced correctly
    gs = GridSpec(2, 4, figure=fig, width_ratios=[48, 48, 0.5, 2.5], height_ratios=[0.85, 0.85], wspace=0.0, hspace=0.0)

    # Create axes for each subplot
    ax1 = fig.add_subplot(gs[0, 0], projection=WCS(hdr).celestial)
    ax2 = fig.add_subplot(gs[0, 1], projection=WCS(hdr).celestial)
    ax3 = fig.add_subplot(gs[1, 0], projection=WCS(hdr).celestial)
    ax4 = fig.add_subplot(gs[1, 1], projection=WCS(hdr).celestial)

    cmap1 = mpl.colormaps.get_cmap('RdBu_r')
    cmap2 = mpl.colormaps.get_cmap('gist_heat_r')
    cmap1.set_bad(color='grey')
    cmap2.set_bad(color='grey')

    im1 = ax1.imshow(RM_G, origin='lower', vmin=-300, vmax=300, cmap=cmap1)
    im2 = ax2.imshow(RM_CG_filt, origin='lower', vmin=-300, vmax=300, cmap=cmap1)
    im3 = ax3.imshow(PI_G, origin='lower', vmin=0, vmax=0.6, cmap=cmap2)
    im4 = ax4.imshow(PI_CG, origin='lower', vmin=0, vmax=0.6, cmap=cmap2)

    # Create colorbars
    cbar2 = fig.colorbar(im2, ax=[ax2], cax=fig.add_subplot(gs[0, 3]), orientation='vertical')
    cbar2.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    cbar2.set_ticks([-200, -100, 0, 100, 200])

    cbar4 = fig.colorbar(im4, ax=[ax4], cax=fig.add_subplot(gs[1, 3]), orientation='vertical')
    cbar4.set_label(r'PI (K)', fontsize=fs)
    cbar4.set_ticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])

    axes = [ax1, ax2, ax3, ax4]
    for i, ax in enumerate(axes):
        ax.set_xlim(WCS(hdr).world_to_pixel(c)[0])
        ax.set_ylim(WCS(hdr).world_to_pixel(c)[1])
        ax.tick_params(axis='both', labelsize=fs)
        ax.set_ylabel('  ', fontsize=fs)
        ax.set_xlabel('  ', fontsize=fs)
        ax.tick_params(axis='both', which='both', width=1, length=3)
        ax.coords[0].set_major_formatter('dd')
        ax.coords[1].set_major_formatter('dd')
        ax.coords[0].set_ticks(l_ticks*u.degree)

        if i in [0, 1]:
            ax.coords[0].set_ticks_visible(False)
            ax.coords[0].set_ticklabel_visible(False)
        if i in [1, 3]:
            ax.coords[1].set_ticks_visible(False)
            ax.coords[1].set_ticklabel_visible(False)

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1)

    for cbar in [cbar2, cbar4]:
        cbar.ax.tick_params(axis='y', which='both', width=1, length=3)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(1)

    fig.text(0.5, 0.04, 'Galactic Longitude', fontsize=fs, ha='center')
    fig.text(0.07, 0.5, 'Galactic Latitude', fontsize=fs, va='center', rotation='vertical')

    plt.savefig('../plots/talks/' + filename + '.png', bbox_inches='tight', pad_inches=0,transparent=False, dpi=80)

    return


In [ ]:
lmin_list = np.arange(52,180.25,0.2)
print(lmin_list.shape)

l_list = np.array(np.arange(52,192,2))
print(l_list)

In [ ]:
#lmin = 52

#for i in range(0,len(lmin_list)):
for i in range(0,1):
    name = 'test_movie_map_'+f"{i+1:03}"
    idxl = np.where((l_list > lmin_list[i]) & (l_list <= 13.5+lmin_list[i]))[0]
    l_ticks = l_list[idxl]
    print(name, lmin_list[i], l_ticks)

    make_4panel_v2(llim = [lmin_list[i]+13.5,lmin_list[i]], blim = [-3,5], filename=name, l_ticks=l_ticks)

In [ ]:
def make_2panel_gmims(llim=[192, 52], blim=[-7, 10], filename='test', l_ticks=[50]):
    
    c = SkyCoord(llim, blim, frame='galactic', unit='deg')
    fs = 16

    fig = plt.figure(figsize=(20, 5))

    gs = GridSpec(2, 3, figure=fig, width_ratios=[98.5, 0.5, 1], height_ratios=[1, 1], wspace=0.0, hspace=0.05)
    ax1 = fig.add_subplot(gs[0, 0], projection=WCS(hdr).celestial)
    ax2 = fig.add_subplot(gs[1, 0], projection=WCS(hdr).celestial)

    cmap = mpl.colormaps.get_cmap('RdBu_r')
    cmap.set_bad(color='grey')

    im1 = ax1.imshow(HBN_RM, origin='lower', vmin=-200, vmax=200, cmap=cmap)
    im2 = ax2.imshow(HBN_FD, origin='lower', vmin=-100, vmax=100, cmap=cmap)

    # Create colorbars
    cbar1 = fig.colorbar(im1, ax=[ax1], cax=fig.add_subplot(gs[0, 2]), orientation='vertical')
    cbar1.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    cbar1.set_ticks([-150,-100,-50, 0, 50, 100, 150])

    cbar2 = fig.colorbar(im2, ax=[ax2], cax=fig.add_subplot(gs[1, 2]), orientation='vertical')
    cbar2.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    cbar2.set_ticks([-75,-50,-25, 0, 25, 50,75])

    axes = [ax1, ax2]
    for i, ax in enumerate(axes):
        ax.set_xlim(WCS(hdr).world_to_pixel(c)[0])
        ax.set_ylim(WCS(hdr).world_to_pixel(c)[1])
        ax.tick_params(axis='both', labelsize=fs)
        ax.set_ylabel('  ', fontsize=fs)
        ax.set_xlabel('  ', fontsize=fs)
        ax.tick_params(axis='both', which='both', width=1, length=3)

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1)

    ax1.coords[0].set_ticks_visible(False)
    ax1.coords[0].set_ticklabel_visible(False)

    ax2.coords[0].set_major_formatter('dd')
    ax1.coords[1].set_major_formatter('dd')
    ax2.coords[1].set_major_formatter('dd')
    ax2.coords[0].set_ticks(l_ticks*u.degree)

    #ax2.set_xlabel('Galactic Longitude', fontsize=fs)
    
    for cbar in [cbar1, cbar2]:
        cbar.ax.tick_params(axis='y', which='both', width=1, length=3)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(1)

    fig.text(0.5, 0.02, 'Galactic Longitude', fontsize=fs, ha='center')
    fig.text(0.09, 0.5, 'Galactic Latitude', fontsize=fs, va='center', rotation='vertical')

    plt.savefig('../plots/talks/' + filename + '.png')

    return


In [ ]:
hdu_HBN_FD = fits.open('/srv/data/cgps-gmims/data_for_paper/single_antenna_peakFD.fits')
HBN_FD = hdu_HBN_FD[0].data

hdu_HBN_RM = fits.open('/srv/data/cgps-gmims/data_for_paper/single_antenna_RM.fits')
HBN_RM = hdu_HBN_RM[0].data

In [ ]:
l_ticks = np.arange(50,200,10)
print(l_ticks)

In [ ]:
make_2panel_gmims(llim=[192, 52], blim=[-7, 10], filename='GMIMS_RM_FD', l_ticks=l_ticks)

In [ ]:
hdu_tadpole = fits.open('/srv/data/chime/tadpole_cutout_Mar2024/RMsynth_400_729/FDF_clean_tot.fits')
cube_tadpole = hdu_tadpole[0].data

wcs = WCS(hdu_tadpole[0].header)

fd_tadpole  = wcs.all_pix2world(0,0,range(wcs.array_shape[0]),0)[2]


In [ ]:
fig, ax = plt.subplots(1,1,figsize=(6,2))

ax.plot(fd_tadpole, cube_tadpole[:,150,134], color='k',linewidth=2)
ax.set_xlim(-100,100)
ax.set_xticks([])
ax.set_yticks([])

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(40,6))
cmap = mpl.colormaps.get_cmap('RdBu_r')
cmap.set_bad(color='grey')
ax.imshow(RM_CG_filt, vmin=-200,vmax=200,cmap=cmap,origin='lower')
plt.savefig('../plots/talks/all_CG_RM.png')

In [ ]:
dir_in = '/srv/aordog/cgps-gmims/cgps-gmims-phd/example_field/'

step_files = ['01_tapered_gmims.fits',
              '02_initial_tapered_gmims.fits',
              '03_deconvolved_gmims.fits', 
              '04_lowpass_filtered_gmims.fits',
              '05_deconvolved_gmims_image.fits',
              '06_gmims_primary_beam_match.fits',
              '07_gmims_stprimary.fits',
              '08_feathered_gmims.fits',
              '09_final_gmims.fits',
              '10_initial_st.fits',
              '11_initial_st_uv.fits',
              '12_feathered_st.fits',
              '13_final_st.fits']


image_plane = []

for i in [0,4,5,8,9,12]:
    field = fits.open(dir_in+step_files[i])
    hdr1 = field[0].header
    data = field[0].data[0,0,:,:]
    image_plane.append(data)
    print(data.shape)
hdr1['NAXIS'] = 2
del hdr1['NAXIS3']
del hdr1['NAXIS4']
del hdr1['CTYPE3']                   
del hdr1['CRVAL3']                    
del hdr1['CRPIX3']                       
del hdr1['CDELT3']             
del hdr1['CROTA3']                     
del hdr1['CTYPE4']              
del hdr1['CRVAL4']  
del hdr1['CRPIX4']                      
del hdr1['CDELT4']                   
del hdr1['CROTA4']
#print(repr(hdr))
wcs = WCS(hdr1)
print(wcs)
print('')

uv_plane = []

for i in [1,2,3,6,7,10,11]:
    field = fits.open(dir_in+step_files[i])
    hdr2 = field[0].header
    data = field[0].data[0,0,:,:]
    uv_plane.append(data)
    print(data.shape)

uvdelt = hdr2['CDELT2']
print(uvdelt)

In [ ]:
fs = 22
bottom = 0.07
top = 0.98
right = 0.96
left = 0.04
PImin = -0.1
PImax = 0.1
cbticks = [-0.1,-0.05,0.,0.05,0.1]
cbticksuv = [0,1,2,3]
numpix = 41
shrink = 1.

axs = ['ax1','ax2','ax3','ax4','ax5']
panels = ['(a)','(b)','(c)','(d)', '(e)']

ext_step = ((numpix-1)/2)*uvdelt+uvdelt/2
extent2=[-ext_step,ext_step,-ext_step,ext_step]

fig = plt.figure(figsize=(20,15))

plt.subplots_adjust(wspace=0.2,hspace=0.3)
plt.subplots_adjust(top = top, bottom = bottom, right = right, left = left)

axs[0] = fig.add_subplot(3,3,1,projection=wcs)
image1 = axs[0].imshow(image_plane[0],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image1, orientation='vertical',shrink=shrink,ax=axs[0],ticks=cbticks)
cb.set_label(label='K',size=fs)
cb.ax.tick_params(labelsize=fs)


axs[1] = fig.add_subplot(3,3,4)
ampl = uv_plane[0][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
print(ampl_new.shape)
print(511-int((numpix-1)/2),511+int((numpix+1)/2),
      511-int((numpix-1)/2),511+int((numpix+1)/2))
image2 = axs[1].imshow(ampl_new[511-int((numpix-1)/2):511+int((numpix+1)/2),
                                511-int((numpix-1)/2):511+int((numpix+1)/2)]/1000,
           vmin=0,vmax=3,origin='lower',cmap='gray',extent=extent2)
cb = fig.colorbar(image2, orientation='vertical',shrink=shrink,ax=axs[1],ticks=cbticksuv)
cb.set_label(label='$\\times$ 10 uv-plane amplitude',size=fs)
cb.ax.tick_params(labelsize=fs)



axs[2] = fig.add_subplot(3,3,5)
ampl = uv_plane[1][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image3 = axs[2].imshow(ampl_new[511-int((numpix-1)/2):511+int((numpix+1)/2),
                                511-int((numpix-1)/2):511+int((numpix+1)/2)]/1000,
           vmin=0,vmax=3,origin='lower',cmap='gray',extent=extent2)
cb = fig.colorbar(image3, orientation='vertical',shrink=shrink,ax=axs[2],ticks=cbticksuv)
cb.set_label(label='$\\times$ 10 uv-plane amplitude',size=fs)
cb.ax.tick_params(labelsize=fs)

axs[3] = fig.add_subplot(3,3,6)
ampl = uv_plane[2][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image4 = axs[3].imshow(ampl_new[511-int((numpix-1)/2):511+int((numpix+1)/2),
                                511-int((numpix-1)/2):511+int((numpix+1)/2)]/1000,
           vmin=0,vmax=3,origin='lower',cmap='gray',extent=extent2)
cb = fig.colorbar(image4, orientation='vertical',shrink=shrink,ax=axs[3],ticks=cbticksuv)
cb.set_label(label='$\\times$ 10 uv-plane amplitude',size=fs)
cb.ax.tick_params(labelsize=fs)


axs[4] = fig.add_subplot(3,3,9,projection=wcs)
image5 = axs[4].imshow(image_plane[1],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image5, orientation='vertical',shrink=shrink,ax=axs[4],ticks=cbticks)
cb.set_label(label='K',size=fs)
cb.ax.tick_params(labelsize=fs)


for i in [1,2,3]:
    axs[i].tick_params(axis="x", labelsize=fs)
    axs[i].tick_params(axis="y", labelsize=fs)
    axs[i].set_xlabel('$u$ (m)',fontsize=fs)
    axs[i].set_ylabel('$v$ (m)',fontsize=fs,labelpad=-5)
    axs[i].set_xticks([-40,-20,0,20,40])
    axs[i].set_yticks([-40,-20,0,20,40])

for i in [0,4]:
    axs[i].coords[0].set_major_formatter('hh:mm')
    axs[i].coords[0].set_ticks([300.,297.5,295.5]*u.degree)
    axs[i].coords[1].set_major_formatter('dd')
    axs[i].coords[0].set_ticklabel(size=fs)
    axs[i].coords[1].set_ticklabel(size=fs)
    axs[i].coords[0].set_axislabel('Right Ascension', fontsize=fs)
    axs[i].coords[1].set_axislabel('Declination', fontsize=fs)

x1 = 0.16
x2 = 0.4855
x3 = 0.81
y1 = 0.3
y2 = 0.33
y3 = 0.7
y4 = 0.67
dax = 0.01
day = 0.015
lw = 2.5
    
plt.plot([x1,x2], [y1,y1], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')
plt.plot([x2,x3], [y3,y3], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')

plt.plot([x1,x1], [y1,y2], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')
plt.plot([x2,x2], [y1,y2], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')

plt.plot([x2,x2], [y3,y4], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')
plt.plot([x3,x3], [y3,y4], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')

plt.plot([x2,x2+dax], [y2,y2-day], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')
plt.plot([x2,x2-dax], [y2,y2-day], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')

plt.plot([x3,x3+dax], [y4,y4+day], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')
plt.plot([x3,x3-dax], [y4,y4+day], lw=lw,transform=gcf().transFigure, clip_on=False,color='black')

plt.text(0.45,0.95,'GMIMS-HBN: Stokes $U$, Band B',transform=gcf().transFigure, clip_on=False,fontsize=fs+4)
plt.text(0.16,0.27,'Divide by single-antenna beam transform',transform=gcf().transFigure, clip_on=False,fontsize=fs)
plt.text(0.6,0.72,'Low pass filter',transform=gcf().transFigure, clip_on=False,fontsize=fs)

plt.savefig('../plots/talks/combining_steps_first_v2_nolabels.png')

In [ ]:
fs = 20
bottom = 0.05
top = 0.96
right = 0.96
left = 0.05
PImin = -0.1
PImax = 0.1
numpix = 81
shrink = 1.
cbticks = [-0.1,-0.05,0.,0.05,0.1]
cbticksuv1 = [0.0,0.5,1.0,1.5,2.0]
cbticksuv2 = [0.0,0.1,0.2,0.3,0.4,0.5]

fig = plt.figure(figsize=(13.2,20))
axs = ['ax1','ax2','ax3','ax4','ax5','ax6','ax7','ax8']

ext_step = ((numpix-1)/2)*uvdelt+uvdelt/2
extent2=[-ext_step,ext_step,-ext_step,ext_step]

plt.subplots_adjust(wspace=0.3,hspace=0.3)
plt.subplots_adjust(top = top, bottom = bottom, right = right, left = left)


# SA image matched to ST primary beam:
axs[0] = fig.add_subplot(4,2,1,projection=wcs)
image1 = axs[0].imshow(image_plane[2],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image1, orientation='vertical',shrink=shrink,ticks=cbticks)
cb.set_label(label='K',size=fs,labelpad=-5)
cb.ax.tick_params(labelsize=fs)

# ST image initial:
axs[1] = fig.add_subplot(4,2,2,projection=wcs)
image2 = axs[1].imshow(image_plane[4],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image2, orientation='vertical',shrink=shrink,ticks=cbticks)
cb.set_label(label='K',size=fs,labelpad=-5)
cb.ax.tick_params(labelsize=fs)


# SA in uv matched to ST primary beam:
axs[2] = fig.add_subplot(4,2,3)
ampl = uv_plane[3][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image3 = axs[2].imshow(ampl_new[511-int((numpix-1)/2):511+int((numpix+1)/2),
                                511-int((numpix-1)/2):511+int((numpix+1)/2)]/1000,
           vmin=0,vmax=2,origin='lower',cmap='gray',extent=extent2)
cb = fig.colorbar(image3, orientation='vertical',shrink=shrink,ticks=cbticksuv1)
cb.set_label(label='$\\times$ 10 uv-plane amplitude',size=fs)
cb.ax.tick_params(labelsize=fs)

# ST in uv original:
axs[3] = fig.add_subplot(4,2,4)
ampl = uv_plane[5][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image4 = axs[3].imshow(ampl_new[511-int((numpix-1)/2):511+int((numpix+1)/2),
                                511-int((numpix-1)/2):511+int((numpix+1)/2)]/1000,
           vmin=0,vmax=0.5,origin='lower',cmap='gray',extent=extent2)
cb = fig.colorbar(image4, orientation='vertical',shrink=shrink,ticks=cbticksuv2)
cb.set_label(label='$\\times$ 10 uv-plane amplitude',size=fs)
cb.ax.tick_params(labelsize=fs)


# SA in uv feathered:
axs[4] = fig.add_subplot(4,2,5)
ampl = uv_plane[4][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image5 = axs[4].imshow(ampl_new[511-int((numpix-1)/2):511+int((numpix+1)/2),
                                511-int((numpix-1)/2):511+int((numpix+1)/2)]/1000,
           vmin=0,vmax=2,origin='lower',cmap='gray',extent=extent2)
cb = fig.colorbar(image5, orientation='vertical',shrink=shrink,ticks=cbticksuv1)
cb.set_label(label='$\\times$ 10 uv-plane amplitude',size=fs)
cb.ax.tick_params(labelsize=fs)

# ST in uv feathered:
axs[5] = fig.add_subplot(4,2,6)
ampl = uv_plane[6][1:1024,0:513]
ampl_refl = np.flip(np.flip(ampl,axis=0),axis=1)
ampl_new = np.concatenate([ampl_refl[:,0:512],ampl],axis=1)[:,1:1024]
image6 = axs[5].imshow(ampl_new[511-int((numpix-1)/2):511+int((numpix+1)/2),
                                511-int((numpix-1)/2):511+int((numpix+1)/2)]/1000,
           vmin=0,vmax=0.5,origin='lower',cmap='gray',extent=extent2)
cb = fig.colorbar(image6, orientation='vertical',shrink=shrink,ticks=cbticksuv2)
cb.set_label(label='$\\times$ 10 uv-plane amplitude',size=fs)
cb.ax.tick_params(labelsize=fs)


# SA final image:
axs[6] = fig.add_subplot(4,2,7,projection=wcs)
image7 = axs[6].imshow(image_plane[3],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image7, orientation='vertical',shrink=shrink,ticks=cbticks)
cb.set_label(label='K',size=fs,labelpad=-5)
cb.ax.tick_params(labelsize=fs)

# ST final image:
axs[7] = fig.add_subplot(4,2,8,projection=wcs)
image8 = axs[7].imshow(image_plane[5],origin='lower',vmin=PImin,vmax=PImax,cmap='gray')
cb = fig.colorbar(image8, orientation='vertical',shrink=shrink,ticks=cbticks)
cb.set_label(label='K',size=fs,labelpad=-5)
cb.ax.tick_params(labelsize=fs)


for i in [2,3,4,5]:
    axs[i].tick_params(axis="x", labelsize=fs)
    axs[i].tick_params(axis="y", labelsize=fs)
    axs[i].set_xlabel('u (m)',fontsize=fs)
    axs[i].set_ylabel('v (m)',fontsize=fs,labelpad=-5)

for i in [0,1,6,7]:
    axs[i].coords[0].set_major_formatter('hh:mm')
    axs[i].coords[0].set_ticks([300.,297.5,295.5]*u.degree)
    axs[i].coords[1].set_major_formatter('dd')
    axs[i].coords[0].set_ticks_position('br')
    axs[i].coords[0].set_ticklabel(size=fs)
    axs[i].coords[1].set_ticklabel(size=fs)
    axs[i].coords[0].set_axislabel('Right Ascension', fontsize=fs)
    axs[i].coords[1].set_axislabel('Declination', fontsize=fs)


axs[0].text(0.075,0.98,'GMIMS-HBN (single-antenna)',fontsize=fs+2,transform=gcf().transFigure)
axs[1].text(0.57,0.98,'DRAO ST (aperture-synthesis)',fontsize=fs+2,transform=gcf().transFigure)   
    
plt.savefig('../plots/talks/combining_steps_second_v2_nolabels.png')

In [ ]:
def make_the_plots(l0=60, b0=0, lwidth=4, filename='test', 
                   PImax=1, RMmax=300, contours=[0.2], srcs=None,
                  lonspace=1, latspace=0.5, circle=False):

    panels = ['(a)','(b)','(c)','(d)', '(e)', '(f)']
    
    bwidth = 3*lwidth/4
    
    llim=[l0+lwidth/2,l0-lwidth/2]
    blim=[b0-bwidth/2,b0+bwidth/2]

    c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
    fs = 24
    fig = plt.figure(figsize=(21.5,10))

    plt.subplots_adjust(hspace=0.05, wspace=0.05, left=0.07, right=0.9, top=0.98, bottom=0.08)

    # PI maps
    
    cmap = mpl.colormaps.get_cmap('gist_heat_r') 
    cmap.set_bad(color='grey')
    axPI1  = fig.add_subplot(231, projection=WCS(hdr).celestial)
    axPI2  = fig.add_subplot(232, projection=WCS(hdr).celestial)
    axPI3  = fig.add_subplot(233, projection=WCS(hdr).celestial)
    imPI1  = axPI1.imshow(PI[1], origin='lower', vmin=0, vmax=PImax, cmap=cmap)
    imPI2  = axPI2.imshow(PI[0], origin='lower', vmin=0, vmax=PImax, cmap=cmap)
    imPI3  = axPI3.imshow(PI[2], origin='lower', vmin=0, vmax=PImax, cmap=cmap)

    cb_axPI = fig.add_axes([0.91, 0.545, 0.02, 0.43])
    cbarPI = fig.colorbar(imPI3, cax=cb_axPI, orientation='vertical')
    cbarPI.set_label(r'PI (K)', fontsize=fs)

    # RM maps
    
    cmap = mpl.colormaps.get_cmap('RdBu_r') 
    cmap.set_bad(color='grey')
    axRM1  = fig.add_subplot(234, projection=WCS(hdr).celestial)
    axRM2  = fig.add_subplot(235, projection=WCS(hdr).celestial)
    axRM3  = fig.add_subplot(236, projection=WCS(hdr).celestial)
    imRM1  = axRM1.imshow(RM[1], origin='lower', vmin=-RMmax, vmax=RMmax, cmap=cmap)
    imRM2  = axRM2.imshow(RM[0], origin='lower', vmin=-RMmax, vmax=RMmax, cmap=cmap)
    imRM3  = axRM3.imshow(RM[2], origin='lower', vmin=-RMmax, vmax=RMmax, cmap=cmap)

    cb_axRM = fig.add_axes([0.91, 0.085, 0.02, 0.43])
    cbarRM = fig.colorbar(imRM3, cax=cb_axRM, orientation='vertical')
    cbarRM.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    
    fig.text(0.008,0.17,'Galactic Latitude',fontsize=fs,rotation='vertical')
    #fig.text(0.008,0.42,'Galactic Latitude',fontsize=fs,rotation='vertical')
    fig.text(0.42,0.01,'Galactic Longitude',fontsize=fs,rotation='horizontal')

    i = 0
    for ax in [axPI1,axPI2,axPI3,axRM1,axRM2,axRM3]:
        ax.set_xlim(WCS(hdr).world_to_pixel(c)[0])
        ax.set_ylim(WCS(hdr).world_to_pixel(c)[1])
        lon = ax.coords[0]
        lat = ax.coords[1]
        if ax in [axPI2,axPI3,axRM2,axRM3]:
            lat.set_ticklabel_visible(False)
        if ax in [axPI1,axPI2,axPI3]:
            lon.set_ticklabel_visible(False)
        ax.tick_params(axis='both', labelsize=fs)
        lon.set_major_formatter('d.d')
        lat.set_major_formatter('d.d')
        lon.set_ticks(spacing=lonspace* u.deg)  # Minor ticks every 1 degree
        lat.set_ticks(spacing=latspace* u.deg)    
        ax.set_ylabel('  ',fontsize=fs)
        ax.set_xlabel('  ',fontsize=fs)
        ax.tick_params(axis='both', which='both', width=2, length=6)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        ax.contour(PI[2], levels=contours, colors='black')

        if ax in [axPI3,axPI2,axRM3,axRM2]:
            beamw = 3/60
        else:
            beamw = 40/60
            
        cx = llim[0]-0.5
        cy = blim[0]+0.5
        rect = Rectangle((cx-0.4, cy-0.4), 0.8, 0.8, linewidth=1, 
                         edgecolor='black', facecolor='white', transform=ax.get_transform('galactic'),alpha=0.5)
        beam = Ellipse((cx, cy), width=beamw, height=beamw, angle=45,
                        edgecolor='black', facecolor='black', lw=1, transform=ax.get_transform('galactic'))
        ax.add_patch(rect)
        ax.add_patch(beam)

        if circle:
            lcen = 158.35
            bcen = 0.2
            radcirc = 0.8
            circ = Ellipse((lcen, bcen), width=radcirc*2, height=radcirc*2, angle=45,
                        edgecolor='yellow', facecolor='None', lw=3, linestyle='dashed',
                        transform=ax.get_transform('galactic'))
            ax.add_patch(circ)


        if srcs != None:
            for ii in range(0,3):
                src_c = SkyCoord(srcs[ii][0], srcs[ii][1], frame=Galactic, unit="deg")
                print(src_c)
                ax.scatter([WCS(hdr).world_to_pixel(src_c)[0]],[WCS(hdr).world_to_pixel(src_c)[1]],
                            s=200, c='yellow', marker='X',edgecolor= "k")
        i = i+1
    
    for cbar in [cbarPI,cbarRM]:
        cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)
    
    plt.savefig('../plots/talks/'+filename+'.png')

    return

In [ ]:
RM = [RM_C_filt,RM_G_filt,RM_CG_filt]
PI = [PI_C,PI_G,PI_CG]


In [ ]:
def make_1panel_linfit(l_pick, b_pick, outname='test.png'):

    fs = 24
    s = 12
    lbd2_ext = np.linspace(0.0435,0.0456,100)
    lbd2_ext_full = np.linspace(0.025,0.055,100)

    fig, axs = plt.subplots(1,1,figsize=(7,7))
    plt.subplots_adjust(left=0.16, bottom=0.15, right=0.95, top=0.98)

    l_idx0 = np.where(abs(l-l_pick)<dlb/2.)[0][0]
    b_idx0 = np.where(abs(b-b_pick)<dlb/2.)[0][0]

    #l_idx0_G = np.where(abs(l_G-l_pick)<dlb_G/2.)[0][0]
    #b_idx0_G = np.where(abs(b_G-b_pick)<dlb_G/2.)[0][0]

    #PA_G_LOS = PA_G_og[:,b_idx0_G,l_idx0_G]

    print(l[l_idx0])
    #print(l_G[l_idx0_G])
    
    print(b[b_idx0])
    #print(b_G[b_idx0_G])
    print('')

    PA_C_fix = []
    PA_G_fix = []
    PA_CG_fix = []
    PA_C_err = []
    PA_G_err = []
    PA_CG_err = []

    for kk in range(0,4):

        PA_C_fix.append(PA_C_list[kk][0].data[b_idx0,l_idx0])
        PA_G_fix.append(PA_G_list[kk][0].data[b_idx0,l_idx0])
        PA_CG_fix.append(PA_CG_list[kk][0].data[b_idx0,l_idx0])

        PA_C_err.append(PA_C_list[kk][1].data[b_idx0,l_idx0])
        PA_G_err.append(PA_G_list[kk][1].data[b_idx0,l_idx0])
        PA_CG_err.append(PA_CG_list[kk][1].data[b_idx0,l_idx0])

    PAint_pt_G = PAint_G[b_idx0,l_idx0]
    PAint_pt_C = PAint_C[b_idx0,l_idx0]
    PAint_pt_CG = PAint_CG[b_idx0,l_idx0]
    RM_pt_G = G_RM[b_idx0,l_idx0]
    RM_pt_C = C_RM[b_idx0,l_idx0]
    RM_pt_CG = CG_RM[b_idx0,l_idx0]
    err_pt_CG = stderr_CG[b_idx0,l_idx0]
    err_pt_G = stderr_G[b_idx0,l_idx0]
    err_pt_C = stderr_C[b_idx0,l_idx0]

    y1 = -90
    y2 = 90

    #axs.axvspan(lbd2[0], lbd2[3], ymin=y1, ymax=y2, alpha=0.5, color='grey')

    print(lbd2)
    print(PA_C_fix)
    print(PA_C_err)

    axs.errorbar(lbd2,np.array(PA_C_fix)*180/np.pi,yerr=np.array(PA_C_err)*180/np.pi,color='C0',fmt="o",
                      capsize=8,elinewidth=4,ms=s)
    axs.errorbar(lbd2,np.array(PA_G_fix)*180/np.pi,yerr=np.array(PA_G_err)*180/np.pi,color='C4',fmt="s",
                      capsize=8,elinewidth=4,ms=s)
    axs.errorbar(lbd2,np.array(PA_CG_fix)*180/np.pi,yerr=np.array(PA_CG_err)*180/np.pi,color='C3',fmt="d",
                      capsize=8,elinewidth=4,ms=s)
    #axs.scatter(lbd2_G,np.array(PA_G_LOS)*180/np.pi,color='k',s=s)
    #axs.scatter(lbd2,  np.array(PA_G_fix)*180/np.pi,color='C4',s=s)
    
    axs.plot(lbd2_ext,PAint_pt_C*180/np.pi+RM_pt_C*lbd2_ext*180/np.pi,color='C0',linewidth=3)
    axs.plot(lbd2_ext,PAint_pt_G*180/np.pi+RM_pt_G*lbd2_ext*180/np.pi,color='C4',linewidth=3)
    axs.plot(lbd2_ext,PAint_pt_CG*180/np.pi+RM_pt_CG*lbd2_ext*180/np.pi,color='C3',linewidth=3)
    
    custom_line1 = Line2D([0], [0], color='C0', marker='o',ms=s,linewidth=3,
                         label=r'RM$_{\mathrm{ST}}$ = '+str(int(np.round(RM_pt_C,0)))+'$\pm$'+str(int(np.round(err_pt_C,0)))+' rad m$^{-2}$')
    custom_line2 = Line2D([0], [0], color='C4', marker='s',ms=s,linewidth=3,
                         label=r'RM$_{\mathrm{HBN}}$ = '+str(int(np.round(RM_pt_G,0)))+'$\pm$'+str(int(np.round(err_pt_G,0)))+' rad m$^{-2}$')
    custom_line3 = Line2D([0], [0], color='C3', marker='d',ms=s,linewidth=3,
                         label=r'RM$_{\mathrm{full}}$ = '+str(int(np.round(RM_pt_CG,0)))+'$\pm$'+str(int(np.round(err_pt_CG,0)))+' rad m$^{-2}$' )

    
    axs.set_xlim(0.0437,0.0456)
    axs.set_xticks([0.044,0.0445,0.045,0.0455])
    axs.set_xticklabels(['0.044','0.0445','0.045','0.0455'])

    axs.set_yticks([-180,-135,-90,-45,0,45,90,135,180])
    axs.set_xlabel(r'$\lambda^2$ (m$^2$)',fontsize=fs)
    axs.set_yticklabels(['-180','-135','-90','-45','0','45','90','135','180'])
    axs.set_ylabel('PA (degrees)',fontsize=fs)

    axs.set_ylim(y1,y2)
    axs.grid()
    axs.tick_params(axis='both', labelsize=fs, left=True, right=True, which='both', width=2, length=6)
    axs.legend(fontsize=fs,handles=[custom_line1,custom_line2,custom_line3])#, loc='upper left')
    for spine in axs.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)

    plt.savefig(outname)

    return

In [ ]:
PA_CG_list = []
PA_G_list = []
PA_C_list = []

band = ['A','B','C','D']
bandlc = ['a','b','c','d']
for i in range(0,4):
    
    print('band '+band[i])
    
    # CGPS + GMIMS (CG)
    directory = '/srv/data/cgps-gmims/conv_regrid/'
    hdu_CG = fits.open(directory+'PA_'+band[i]+'_CG_conv4_regrd.fits')
    PA_CG_list.append(hdu_CG)
    
    # Grab a header (all the same for now)
    hdr = hdu_CG[0].header
    
    # CGPS only (C)
    directory = '/srv/data/cgps-gmims/conv_regrid/'
    hdu_C = fits.open(directory+'PA_'+band[i]+'_C_conv4_regrd.fits')
    PA_C_list.append(hdu_C)
    
    # GMIMS only (G)
    directory = '/srv/data/cgps-gmims/conv_regrid/'
    hdu_G = fits.open(directory+'PA_'+band[i]+'_G_regrd.fits')
    PA_G_list.append(hdu_G)
    
gc.collect()

In [ ]:
dir_in  = '/srv/data/cgps-gmims/conv_regrid/'

hdu_CG_RM = fits.open(dir_in+'RM_CG_conv4_regrd.fits')

CG_RM     = hdu_CG_RM[0].data
PAint_CG  = hdu_CG_RM[1].data
rvalue_CG = hdu_CG_RM[2].data
stderr_CG = hdu_CG_RM[4].data

CG_RM[hdu_CG_RM[0].data==0] = np.nan
PAint_CG[hdu_CG_RM[0].data==0] = np.nan
rvalue_CG[hdu_CG_RM[0].data==0] = np.nan
stderr_CG[hdu_CG_RM[0].data==0] = np.nan

print(CG_RM.shape)

##########################################################

hdu_G_RM = fits.open(dir_in+'RM_G_regrd.fits')

G_RM     = hdu_G_RM[0].data
PAint_G  = hdu_G_RM[1].data
rvalue_G = hdu_G_RM[2].data
stderr_G = hdu_G_RM[4].data

G_RM[hdu_G_RM[0].data==0] = np.nan
PAint_G[hdu_G_RM[0].data==0] = np.nan
rvalue_G[hdu_G_RM[0].data==0] = np.nan
stderr_G[hdu_G_RM[0].data==0] = np.nan

print(G_RM.shape)

##########################################################

hdu_C_RM = fits.open(dir_in+'RM_C_conv4_regrd.fits')

C_RM     = hdu_C_RM[0].data
PAint_C  = hdu_C_RM[1].data
rvalue_C = hdu_C_RM[2].data
stderr_C = hdu_C_RM[4].data

C_RM[hdu_C_RM[0].data==0] = np.nan
PAint_C[hdu_C_RM[0].data==0] = np.nan
rvalue_C[hdu_C_RM[0].data==0] = np.nan
stderr_C[hdu_C_RM[0].data==0] = np.nan

print(C_RM.shape)

In [ ]:
hdu_G_Q = fits.open('/srv/data/cgps-gmims/gmims_OG/gmims_hbn_Q.fits')
hdu_G_U = fits.open('/srv/data/cgps-gmims/gmims_OG/gmims_hbn_U.fits')

hdr_G_og = hdu_G_Q[0].header
#print(repr(hdr_G_og))

Q_G = hdu_G_Q[0].data
U_G = hdu_G_U[0].data

PA_G_og = 0.5*np.arctan2(U_G,Q_G)

print(PA_G_og.shape)

In [ ]:
freq = np.array([1406.9,1413.8,1427.4,1434.3])
lbd2 = ((3e8)/(freq*1e6))**2

l = WCS(hdr).all_pix2world(range(PA_CG_list[0][0].data.shape[1]) ,0, 0)[0]
b = WCS(hdr).all_pix2world(0, range(PA_CG_list[0][0].data.shape[0]), 0)[1]

dlb = l[0]-l[1]

freq_G = WCS(hdr_G_og).all_pix2world(0, 0, range(Q_G.shape[0]), 0)[2]
lbd2_G = ((3e8)/freq_G)**2

l_G = WCS(hdr_G_og).all_pix2world(range(Q_G.shape[2]) ,0, 0, 0)[0]
b_G = WCS(hdr_G_og).all_pix2world(0, range(Q_G.shape[1]), 0, 0)[1]

dlb_G = l_G[0]-l_G[1]
print(dlb_G)


In [ ]:
make_the_plots(l0=157.5, b0=0.0, lwidth=4.2, filename='158_0_Sh216_PN_talk_RMonly_3points', 
               PImax=0.8, RMmax=150, contours=[0.3], srcs=[[158.75,0.78],[157.67,0.43],[158.46, 0.05]],circle=False)

#make_the_plots(l0=157.5, b0=0.0, lwidth=4.2, filename='158_0_Sh216_PN_talk', 
#               PImax=0.8, RMmax=150, contours=[0.3], src=None,circle=False)

In [ ]:
make_1panel_linfit(157.67, 0.43, '../plots/talks/linfit_pt2.png')

In [ ]:
def make_1panel_linfit_gmims(l_pick, b_pick, outname='test.png'):

    fs = 24
    s = 12
    lbd2_ext = np.linspace(0.0435,0.0456,100)
    lbd2_ext_full = np.linspace(0.025,0.055,100)

    lbd2min = (3e8/((1434.3+3)*1e6))**2
    lbd2max = (3e8/((1406.9-3)*1e6))**2

    print(lbd2min,lbd2max)

    fig, axs = plt.subplots(1,1,figsize=(10,6))
    plt.subplots_adjust(left=0.15, bottom=0.15, right=0.95, top=0.85)

    l_idx0 = np.where(abs(l-l_pick)<dlb/2.)[0][0]
    b_idx0 = np.where(abs(b-b_pick)<dlb/2.)[0][0]

    l_idx0_G = np.where(abs(l_G-l_pick)<dlb_G/2.)[0][0]
    b_idx0_G = np.where(abs(b_G-b_pick)<dlb_G/2.)[0][0]

    PA_G_LOS = PA_G_og[:,b_idx0_G,l_idx0_G]

    print(l_G[l_idx0_G])
    print(b_G[b_idx0_G])
    print('')

    PA_G_fix = []
    PA_G_err = []

    for kk in range(0,4):

        PA_G_fix.append(PA_G_list[kk][0].data[b_idx0,l_idx0])
        PA_G_err.append(PA_G_list[kk][1].data[b_idx0,l_idx0])

    PAint_pt_G = PAint_G[b_idx0,l_idx0]
    RM_pt_G = G_RM[b_idx0,l_idx0]
    err_pt_G = stderr_G[b_idx0,l_idx0]

    axs.axvspan(lbd2min, lbd2max, ymin=-90, ymax=90, alpha=0.5, color='grey')
    axs.scatter(lbd2_G,np.array(PA_G_LOS)*180/np.pi,color='k',s=s)
    axs.scatter(lbd2,  np.array(PA_G_fix)*180/np.pi,color='C4',s=s)
    axs.plot(lbd2_ext_full, PAint_pt_G*180/np.pi + RM_pt_G*lbd2_ext_full*180/np.pi,color='C4',linewidth=3)

    lbd2_ticks = np.arange(0.03,0.06,0.005)
    print(lbd2_ticks)
    freq_ticks = np.round((3e8)/np.sqrt(lbd2_ticks)/1e9,2)
    freq_ticks_str = []
    for i in range(0,6):
        freq_ticks_str.append(f'{freq_ticks[i]:3.2f}')
        #freq_ticks_str.append(str(freq_ticks[i]))
    
    axs.set_xlim(0.03,0.055)
    axs.set_xticks(lbd2_ticks)
    axs.set_ylim(-90,90)
    axs.set_yticks([-90,-45,0,45,90])
    axs.set_xlabel(r'$\lambda^2$ (m$^2$)',fontsize=fs)
    axs.set_ylabel('PA (degrees)',fontsize=fs)

    axtop = axs.twiny()
    axtopticks = axtop.get_xticks()
    print(axtopticks)
    axtop.set_xticks(axtopticks, labels=freq_ticks_str)
    axtop.tick_params(axis='both', labelsize=fs,  which='both', width=2, length=6)
    axtop.set_xlabel(r'$\nu$ (GHz)',fontsize=fs)

    axs.grid()
    axs.tick_params(axis='both', labelsize=fs, left=True, right=True, which='both', width=2, length=6)
    for spine in axs.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)

    plt.savefig(outname)

    return

In [ ]:
make_1panel_linfit_gmims(158.75,0.78, '../plots/talks/full_gmims_linfit_example.png')

In [ ]:
lbd2_ticks = np.arange(0.03,0.06,0.005)
freq_ticks = (3e8)/np.sqrt(lbd2_ticks)/1e6

print(lbd2_ticks)
print(freq_ticks)

In [ ]:
freq_labels = str(freq_ticks)

In [ ]:
print(freq_labels)

In [ ]:
((3e8)/(1279e6))**2